# AGN point-source detectability from an astromodels YAML

Test the catalog flux of each AGN independently over **0.1–10 MeV (100–10,000 keV)**.
This extends [PointSource_Sensitivity.ipynb](PointSource_Sensitivity.ipynb) to cutoff
power laws loaded from a combined YAML. The defaults are **all 13 sources and 20
Poisson realizations per source**, using the DC4 three-month continuum response,
background, and spacecraft history. A separate two-year result is an approximate
extrapolation, not a two-year simulation.

Each trial fits source normalization and background rate, keeping position and
spectral shape fixed. Other AGNs in the YAML are **not** included in that trial:
this is isolated-source detectability, not a simultaneous sky fit. These tests do
not search for the minimum detectable flux. Twenty trials provide only an
exploratory estimate of median TS; increase the trial count for stable results.

## Generate the input model

In the Python environment containing astromodels and scipy, run:

```bash
cd /path/to/AGN-HE-detectability
python generate_cosi_agn_model.py
# To regenerate an existing model: add --overwrite
```

The generator reads `AGN_Index_ecut_scatter_cosi_detectable.csv` and the local
`Swift_BAT_105mo_catalog.csv`, obtains counterpart positions, and saves
`cosi_agn_models.yaml`. Set `MODEL_PATH` below if your repositories are not siblings.
The original CSV and its 0.2–5 MeV flux columns are not needed at notebook runtime.

For photon spectrum $N(E)=K(E/1\,\mathrm{keV})^{-\Gamma}\exp(-E/E_c)$,
`index = -Gamma`, `xc = Ecut_keV`, and `K = norm_at_1_keV / 1.602176634e-9`.
The CSV normalization is an **energy-flux coefficient**, not a photon normalization.
BAT 14–195 keV normalization is preserved; we integrate the loaded spectra anew
over **100–10,000 keV**. This changes the reporting band without renormalizing the source.

Use a Python 3.12+ kernel with this cosipy checkout and its dependencies installed.
The continuum response is coarse; these estimates inherit its limitations and the
assumed background model. Large response downloads and source-response calculations
can take substantial time and memory.


In [ ]:
import os
# Set before importing numerical libraries.
for variable in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[variable] = "1"

import hashlib
import json
import logging
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import astropy.units as u
from IPython.display import display
from scipy.integrate import quad
from astromodels import Cutoff_powerlaw, Parameter, clone_model, load_model
from astromodels.utils.configuration import astromodels_config
from threeML import DataList, JointLikelihood
from histpy import Histogram

from cosipy.spacecraftfile import SpacecraftHistory
from cosipy.response import (
    FullDetectorResponse, BinnedInstrumentResponse,
    BinnedThreeMLPointSourceResponse, BinnedThreeMLModelFolding,
)
from cosipy.data_io import EmCDSBinnedData
from cosipy.background_estimation import FreeNormBinnedBackground
from cosipy.statistics import PoissonLikelihood
from cosipy.interfaces import ThreeMLPluginInterface
from cosipy.util import fetch_wasabi_file

logging.getLogger("cosipy").setLevel(logging.WARNING)


In [ ]:
# Works from the cosipy repository root or any directory beneath it.
COSI_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "cosipy").is_dir() and (p / "pyproject.toml").is_file()),
    Path.cwd(),
)
MODEL_PATH = COSI_ROOT.parent / "AGN-HE-detectability" / "cosi_agn_models.yaml"
# Reuse the user's existing DC4 files without writing into the OneDrive folder.
DATA_DIR = (Path.home() / "Library/CloudStorage/OneDrive-ClemsonUniversity/COSI/"
            "Radio_Quiet_AGN/DC4_Files")
DOWNLOAD_DIR = COSI_ROOT / "agn_sensitivity_data"  # Only missing reference files.
OUTPUT_DIR = COSI_ROOT / "agn_sensitivity_results"
SOURCE_NAMES = None  # None = all sources; e.g. ["CenA", "NGC2110"] for a subset.
N_REALIZATIONS = 20
RANDOM_SEED = 20260916
DOWNLOAD_DATA = False  # All three files are now in DATA_DIR; True fetches missing files.
ENERGY_BAND_KEV = (100.0, 10000.0)
TS_THRESHOLD = 9.0
EXPOSURE_SCALE_2YR = 8.0  # 24 months / nominal 3-month DC4 history.
BACKGROUND_FLOOR_COUNTS = 1e-12  # Applied only to exactly empty background bins.

if not MODEL_PATH.is_file():
    raise FileNotFoundError(f"Generate the YAML first or update MODEL_PATH: {MODEL_PATH}")
if not isinstance(N_REALIZATIONS, int) or N_REALIZATIONS < 1:
    raise ValueError("N_REALIZATIONS must be a positive integer")
if not isinstance(RANDOM_SEED, int) or RANDOM_SEED < 0:
    raise ValueError("RANDOM_SEED must be a nonnegative integer")
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## Load and inspect the catalog spectra

All reported photon/energy fluxes below cover **0.1–10 MeV**. `K` is the coefficient
of the cutoff power law at a 1 keV pivot (the exponential still multiplies it).
Only the fitted model copies are changed; the loaded catalog model stays unchanged.


In [ ]:
KEV_TO_ERG = 1.602176634e-9


def band_fluxes(shape, band=(100.0, 10000.0)):
    photon = quad(lambda e: float(shape(e)), *band, epsabs=1e-30, epsrel=1e-9)[0]
    energy = quad(lambda e: e * float(shape(e)), *band, epsabs=1e-30, epsrel=1e-9)[0]
    return photon, energy * KEV_TO_ERG


def select_sources(catalog, names=None):
    available = list(catalog.point_sources)
    if catalog.extended_sources or not available:
        raise ValueError("Expected a YAML containing only point sources")
    selected = available if names is None else list(names)
    if not selected or len(selected) != len(set(selected)):
        raise ValueError("Choose at least one source, without duplicates")
    unknown = set(selected) - set(available)
    if unknown:
        raise ValueError(f"Unknown sources: {sorted(unknown)}; available: {available}")
    for name in selected:
        source = catalog.point_sources[name]
        if len(source.components) != 1 or not isinstance(source.spectrum.main.shape, Cutoff_powerlaw):
            raise ValueError(f"{name}: expected one Cutoff_powerlaw component")
        shape = source.spectrum.main.shape
        if not np.all(np.isfinite([shape.K.value, shape.index.value, shape.xc.value, shape.piv.value])):
            raise ValueError(f"{name}: nonfinite spectral parameters")
        if shape.K.value <= 0 or shape.xc.value <= 0 or shape.piv.value <= 0:
            raise ValueError(f"{name}: K, xc and piv must be positive")
    return selected


def isolated_model(catalog, name):
    model = clone_model(catalog)
    for other in list(model.point_sources):
        if other != name:
            model.remove_source(other)
    for parameter in model.parameters.values():
        parameter.free = False
    shape = model.point_sources[name].spectrum.main.shape
    # YAML loading restores the function's default log transform. Remove it here,
    # so the nested background-only hypothesis can have EXACTLY zero source flux.
    shape.K.remove_transformation()
    shape.K.min_value = 0.0
    shape.K.max_value = None
    shape.K.delta = 0.1 * shape.K.value
    shape.K.free = True
    return model


def source_table(catalog, names, band):
    rows = []
    for name in names:
        source = catalog.point_sources[name]
        shape = source.spectrum.main.shape
        photon, energy = band_fluxes(shape, band)
        rows.append({
            "source": name, "ra_deg": source.position.ra.value,
            "dec_deg": source.position.dec.value, "gamma": -shape.index.value,
            "ecut_keV": shape.xc.value, "K_ph_cm2_s_keV": shape.K.value,
            "band_min_keV": band[0], "band_max_keV": band[1],
            "photon_flux_0p1_10_MeV_ph_cm2_s": photon,
            "energy_flux_0p1_10_MeV_erg_cm2_s": energy,
        })
    return pd.DataFrame(rows)


In [ ]:
catalog_model = load_model(str(MODEL_PATH))
selected_names = select_sources(catalog_model, SOURCE_NAMES)
catalog_table = source_table(catalog_model, selected_names, ENERGY_BAND_KEV)
display(catalog_table)


## Obtain the DC4 continuum files

These are the same continuum choices and checksums provided in the original point-source
tutorial. Existing matching files are reused. The background histogram represents counts
for the supplied three-month history; its fitted normalization is a rate in Hz.

The default `DATA_DIR` is the supplied OneDrive `DC4_Files` directory. The response,
15-second orientation, and downloaded smoothed continuum background there match
the reference checksums. Its
`Background/Total_DC4_BG_3months_binned_data_filtered_with_SAAcut_withSAAbck.hdf5`
is a different, time-binned background simulation. It is **not substituted** for
the reference smoothed background. With `DOWNLOAD_DATA = True`, missing reference
files are downloaded to `DOWNLOAD_DIR` in the cosipy workspace. The notebook only
reads files already present in `DATA_DIR`.


In [ ]:
FILES = {
    "orientation": (
        "Orientation/DC4_final_530km_3_month_with_slew_15sbins_GalacticEarth_SAA.fits",
        "ca94ff1d7a73c1f41479aaf598807673",
    ),
    "response": (
        "Responses/ResponseContinuum.o3.e100_10000.b10log.s10396905069491.m2284.filtered.nonsparse.binnedimaging.imagingresponse.h5",
        "7121f094be50e7bfe9b31e53015b0e85",
    ),
    "background": (
        "Backgrounds/gal_DC4_bkg_continuum_binned_smoothed_fwhm_10.0deg.hdf5",
        "1acceb619b0d7e43eea8cb29ffe74279",
    ),
}
paths = {}
for key, (remote, _) in FILES.items():
    filename = Path(remote).name
    candidates = (DATA_DIR / filename, DATA_DIR / "Background" / filename,
                  DOWNLOAD_DIR / filename)
    paths[key] = next((path for path in candidates if path.is_file()), DOWNLOAD_DIR / filename)
    print(f"{key}: {paths[key]}")
for key, (remote, checksum) in FILES.items():
    if DOWNLOAD_DATA:
        fetch_wasabi_file("COSI-SMEX/DC4/Data/" + remote,
                          output=paths[key], checksum=checksum)
    elif not paths[key].is_file():
        raise FileNotFoundError(paths[key])
    else:
        # Streaming avoids loading a large response into memory just to hash it.
        with paths[key].open("rb") as handle:
            actual = hashlib.file_digest(handle, "md5").hexdigest()
        if actual != checksum:
            raise ValueError(f"Checksum mismatch: {paths[key]}")


## Check the response and background

Use all native bins, without an additional measured-energy cut. Both incident and
measured energy coverage are checked against **100–10,000 keV**. No partial-bin
rescaling is applied. Scattering-angle bins must also agree; cosipy handles the
rotation from spacecraft coordinates to the background's sky coordinates.

Exactly empty background bins require care in a Poisson likelihood. We apply the
explicit, tiny floor configured above to those bins, using the same template for
simulation and fitting. This is a numerical convention, **not a measured physical
background**. Signal in such bins can inflate TS; the summary reports its expected
counts and the number of floored bins so these cases can be inspected.


In [ ]:
def regularize_background(histogram, floor):
    background = histogram.project("Em", "Phi", "PsiChi").to_dense()
    values = np.asarray(background.contents, dtype=float).copy()
    if not np.all(np.isfinite(values)) or np.any(values < 0) or values.sum() <= 0:
        raise ValueError("Background must contain finite nonnegative counts with positive total")
    if not np.isfinite(floor) or floor <= 0:
        raise ValueError("The background floor must be finite and positive")
    empty = values == 0
    values[empty] = floor
    background[:] = values
    return background, empty


def check_response_axes(detector_response, background, band):
    for label, axis in (("Ei", detector_response.axes["Ei"]),
                        ("Em", detector_response.measurement_axes["Em"]),
                        ("background Em", background.axes["Em"])):
        edges = axis.edges.to_value(u.keV)
        if not np.all(np.diff(edges) > 0) or not np.allclose(edges[[0, -1]], band):
            raise ValueError(f"{label} must span {band} keV; got {edges[[0, -1]]}")
    for label in ("Em", "Phi"):
        if background.axes[label] != detector_response.measurement_axes[label]:
            raise ValueError(f"Background/response {label} axes do not match")
    if background.axes["PsiChi"].coordsys is None:
        raise ValueError("Background PsiChi axis needs a coordinate system")


def make_folding(model, detector_response, background, orientation):
    # Only axes enter the response construction. Trial counts go into a fresh
    # PoissonLikelihood; no private response data attributes need to be changed.
    template_data = EmCDSBinnedData(background)
    instrument = BinnedInstrumentResponse(detector_response, template_data)
    psr = BinnedThreeMLPointSourceResponse(
        data=template_data, instrument_response=instrument, sc_history=orientation,
        energy_axis=detector_response.axes["Ei"],
        polarization_axis=(detector_response.axes["Pol"]
                           if "Pol" in detector_response.axes.labels else None),
        nside=2 * background.axes["PsiChi"].nside,
    )
    folding = BinnedThreeMLModelFolding(data=template_data, point_source_response=psr)
    folding.set_model(model)
    signal = folding.expectation()
    if signal.axes != background.axes:
        raise ValueError("Folded signal and background axes do not agree")
    values = np.asarray(signal.contents, dtype=float)
    if not np.all(np.isfinite(values)) or np.any(values < 0) or values.sum() <= 0:
        raise ValueError("Folded signal must be finite, nonnegative and nonzero")
    return folding, signal


In [ ]:
sc_orientation = SpacecraftHistory.open(paths["orientation"])
background, empty_background_bins = regularize_background(
    Histogram.open(paths["background"]), BACKGROUND_FLOOR_COUNTS,
)
livetime_s = float(sc_orientation.cumulative_livetime().to_value(u.s))
if not np.isfinite(livetime_s) or livetime_s <= 0:
    raise ValueError("Spacecraft history must have positive finite livetime")
background_rate_hz = float(np.asarray(background.contents).sum()) / livetime_s
with FullDetectorResponse.open(paths["response"]) as detector_response:
    check_response_axes(detector_response, background, ENERGY_BAND_KEV)
print(f"Livetime: {livetime_s:.3f} s; background rate: {background_rate_hz:.6g} Hz")
print(f"Empty background bins receiving a floor: {empty_background_bins.sum()}")
print("Likelihood and flux-reporting range: 0.1–10 MeV")


## Independent Poisson trials and likelihood fits

The deterministic source expectation is folded once per source and reused for injection.
Sampling it is equivalent to drawing a new source-injected realization with fixed
position and spectrum. The fit reuses the cached point-source response.

For each realization, fit the alternative with `K >= 0` and a free background rate,
then refit the background with `K = 0` fixed. Record
$TS=2[\log L(\mathrm{source+background})-\log L(\mathrm{background})]$.
Negative TS beyond a small numerical tolerance and nonfinite likelihoods are failures.
The fit wrapper temporarily disables parameter-transform checks while threeML
constructs its result copies, restoring the prior setting even after an exception.
This permits the exact zero normalization with astromodels 2.5.1.

Failed trials are saved, not replaced by zero TS, and summaries report their number.
A failed trial makes the source's detection classification inconclusive until rerun.

At a preselected position and fixed spectral shape, $Z\approx\sqrt{TS}$ is the usual
one-sided asymptotic approximation for a nonnegative signal amplitude. It is not an
exact finite-count calibration or a correction for searching many sources. We use
median TS >= 9 as an approximate local 3-sigma criterion.


In [ ]:
def draw_trial(signal, background, rng):
    if signal.axes != background.axes:
        raise ValueError("Signal/background axes do not match")
    counts = rng.poisson(np.asarray(signal.contents, dtype=float))
    counts += rng.poisson(np.asarray(background.contents, dtype=float))
    histogram = Histogram(background.axes)
    histogram[:] = counts
    return EmCDSBinnedData(histogram)


def checked_ts(loglike_signal, loglike_null, tolerance=1e-4):
    if not np.all(np.isfinite([loglike_signal, loglike_null])):
        raise ValueError("Nonfinite fitted likelihood")
    ts_raw = 2 * (loglike_signal - loglike_null)
    if ts_raw < -tolerance:
        raise ValueError(f"Materially negative TS: {ts_raw:.6g}")
    return max(0.0, ts_raw), ts_raw


def fit_without_transforms(analysis):
    # threeML clones the fitted model when building its results. astromodels
    # restores Cutoff_powerlaw's default log transform in that clone, which
    # otherwise rejects our zero lower bound even after remove_transformation().
    # Disable transformation checks during fitting/result construction only;
    # restore the user's setting even if a fit raises. Run trials sequentially.
    previous = astromodels_config.modeling.use_parameter_transforms
    try:
        astromodels_config.modeling.use_parameter_transforms = False
        # A results-only clone may internally represent exact zero as log10(0).
        # Suppress only that known warning; retain all other numerical warnings.
        with warnings.catch_warnings():
            warnings.filterwarnings("ignore", message="divide by zero encountered in _log10",
                                    category=RuntimeWarning)
            analysis.fit(compute_covariance=False, quiet=True, n_samples=1)
    finally:
        astromodels_config.modeling.use_parameter_transforms = previous


def fit_trial(model, folding, signal, background, orientation, rate_hz, rng):
    source = next(iter(model.point_sources.values()))
    shape = source.spectrum.main.shape
    catalog_K = shape.K.value
    result = {"status": "failed", "error": "", "ts": np.nan, "ts_raw": np.nan,
              "loglike_signal": np.nan, "loglike_null": np.nan,
              "K_fit_ph_cm2_s_keV": np.nan, "background_rate_fit_hz": np.nan,
              "background_rate_null_hz": np.nan}
    plugin = None
    try:
        shape.K.value, shape.K.free = catalog_K, True
        data = draw_trial(signal, background, rng)
        bkg = FreeNormBinnedBackground({"Background": background},
                                      sc_history=orientation, copy=True)
        likelihood = PoissonLikelihood(data, folding, bkg)
        plugin = ThreeMLPluginInterface("cosi", likelihood, folding, bkg)
        # FreeNormBinnedBackground normalizes the template to unit integral.
        # Its amplitude is counts/livetime in Hz, not a dimensionless scale.
        plugin.bkg_parameter["Background"] = Parameter(
            "Background", rate_hz, min_value=0, max_value=None,
            delta=0.05 * rate_hz, unit=u.Hz,
        )
        analysis = JointLikelihood(model, DataList(plugin), verbose=False)
        analysis.set_minimizer("minuit")
        fit_without_transforms(analysis)
        result["loglike_signal"] = float(plugin.get_log_like())
        result["K_fit_ph_cm2_s_keV"] = shape.K.value
        result["background_rate_fit_hz"] = plugin.bkg_parameter["Background"].value

        shape.K.value, shape.K.free = 0.0, False
        fit_without_transforms(analysis)
        result["loglike_null"] = float(plugin.get_log_like())
        result["background_rate_null_hz"] = plugin.bkg_parameter["Background"].value
        result["ts"], result["ts_raw"] = checked_ts(
            result["loglike_signal"], result["loglike_null"],
        )
        result["status"] = "ok"
    except Exception as exc:
        result["error"] = f"{type(exc).__name__}: {exc}"
    finally:
        shape.K.value, shape.K.free = catalog_K, True
        # JointLikelihood attaches its nuisance parameters to the model.
        # Remove them so the next trial starts with a clean source-only model.
        if plugin is not None:
            for parameter in plugin.nuisance_parameters.values():
                if parameter.name in model:
                    model.remove_external_parameter(parameter.name)
    return result


def summarize_trials(trials, flux_table, exposure_scale=8.0, threshold=9.0):
    summaries = []
    for source in flux_table.to_dict("records"):
        group = trials.loc[trials["source"] == source["source"]]
        valid = group.loc[(group["status"] == "ok") & np.isfinite(group["ts"]), "ts"]
        median = float(valid.median()) if len(valid) else np.nan
        failures = len(group) - len(valid)
        complete = len(group) > 0 and failures == 0
        summaries.append({
            **source, "n_trials": len(group), "n_valid": len(valid), "n_failed": failures,
            "ts_threshold": threshold, "median_ts_3mo": median, "approx_local_sigma_3mo": np.sqrt(median),
            "median_ts_2yr_approx": exposure_scale * median,
            "approx_local_sigma_2yr": np.sqrt(exposure_scale * median),
            "passes_ts_threshold_3mo": bool(median >= threshold) if complete else None,
            "passes_ts_threshold_2yr_approx": bool(exposure_scale * median >= threshold) if complete else None,
            "classification_status": "complete" if complete else "inconclusive_fit_failures",
        })
    return pd.DataFrame(summaries)


In [ ]:
# Outputs are checkpointed after each source; rerunning replaces these result files.
run_metadata = {
    "model_path": str(MODEL_PATH.resolve()),
    "model_sha256": hashlib.sha256(MODEL_PATH.read_bytes()).hexdigest(),
    "source_names": selected_names, "n_realizations": N_REALIZATIONS,
    "random_seed": RANDOM_SEED, "energy_band_keV": ENERGY_BAND_KEV,
    "livetime_s": livetime_s, "nominal_duration_months": 3,
    "exposure_scale_2yr": EXPOSURE_SCALE_2YR, "ts_threshold": TS_THRESHOLD,
    "background_floor_counts": BACKGROUND_FLOOR_COUNTS,
    "background_rate_hz": background_rate_hz,
    "files": {key: {"path": str(paths[key]), "md5": checksum}
              for key, (_, checksum) in FILES.items()},
}
(OUTPUT_DIR / "run_metadata.json").write_text(json.dumps(run_metadata, indent=2))
all_trials, diagnostics = [], []
with FullDetectorResponse.open(paths["response"]) as detector_response:
    check_response_axes(detector_response, background, ENERGY_BAND_KEV)
    for name in selected_names:
        print(f"Preparing {name} ({N_REALIZATIONS} trials)...", flush=True)
        model = isolated_model(catalog_model, name)
        folding, signal = make_folding(model, detector_response, background, sc_orientation)
        source_counts = np.asarray(signal.contents, dtype=float)
        in_empty = float(source_counts[empty_background_bins].sum())
        diagnostics.append({
            "source": name, "expected_source_counts_3mo": float(source_counts.sum()),
            "n_floored_background_bins": int(empty_background_bins.sum()),
            "expected_signal_counts_in_floored_bins": in_empty,
        })
        if in_empty > 0.1:
            print(f"  Review background: {in_empty:.3g} expected signal counts in floored bins.")
        # Use the source's index in the original catalog, so selecting a subset
        # does not change its random stream.
        source_index = list(catalog_model.point_sources).index(name)
        rng = np.random.default_rng(np.random.SeedSequence([RANDOM_SEED, source_index]))
        for trial in range(N_REALIZATIONS):
            result = fit_trial(model, folding, signal, background, sc_orientation,
                               background_rate_hz, rng)
            all_trials.append({"source": name, "trial": trial, **result})
            print(f"  {trial + 1}/{N_REALIZATIONS}: {result['status']}; TS={result['ts']:.3g}", flush=True)
            if result["status"] != "ok":
                print("   ", result["error"])
        trials = pd.DataFrame(all_trials)
        trials.to_csv(OUTPUT_DIR / "agn_trials_0p1_10_MeV.csv", index=False)
        completed_fluxes = catalog_table[catalog_table["source"].isin(trials["source"])]
        summary = summarize_trials(trials, completed_fluxes, EXPOSURE_SCALE_2YR, TS_THRESHOLD)
        summary = summary.merge(pd.DataFrame(diagnostics), on="source", validate="one_to_one")
        summary.to_csv(OUTPUT_DIR / "agn_summary_0p1_10_MeV.csv", index=False)
        del folding, signal, model

display(summary.sort_values("median_ts_3mo", ascending=False))
print(f"Saved results to {OUTPUT_DIR.resolve()}")


## TS distributions and approximate two-year significance

For steady flux and comparable observing efficiency and background conditions,
$TS$ approximately grows with exposure in the background-dominated regime:

$$TS_{2\,\mathrm{yr}}\approx8\,TS_{3\,\mathrm{mo}},\qquad Z\approx\sqrt{TS}.$$

The factor eight uses nominal calendar durations; the three-month calculation
itself uses the spacecraft history's actual livetime. This extrapolation is not
a full two-year orbit simulation and can fail with source variability, changing
background, sparse counts, or systematic errors. No flux threshold has been found:
a catalog flux exceeding median TS = 9 does not make that flux the sensitivity limit.
Rows with failed fits remain inconclusive; their displayed medians use successful
trials only. Inspect the failure counts and floored-background diagnostics before
interpreting a result. No trials are silently replaced or retried.


In [ ]:
ncols = 3
nrows = (len(selected_names) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(14, 3.2 * nrows), squeeze=False)
for ax, name in zip(axes.flat, selected_names):
    group = trials.loc[trials["source"] == name]
    ts = group.loc[group["status"] == "ok", "ts"].dropna()
    if len(ts):
        ax.hist(ts, bins=min(10, len(ts)), alpha=0.75)
        ax.axvline(ts.median(), color="tab:red", label=f"median {ts.median():.2f}")
    ax.axvline(TS_THRESHOLD, color="black", linestyle="--", label="TS = 9")
    ax.set(title=f"{name}: {len(ts)}/{len(group)} valid", xlabel="TS (three months)", ylabel="Trials")
    ax.legend(fontsize=8)
for ax in list(axes.flat)[len(selected_names):]:
    ax.set_visible(False)
fig.suptitle("AGN catalog-flux tests, 0.1–10 MeV", y=1.01)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "agn_ts_distributions_0p1_10_MeV.png", dpi=160, bbox_inches="tight")
plt.show()

ordered = summary.sort_values("median_ts_3mo", ascending=False)
fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(ordered))
ax.plot(x, ordered["approx_local_sigma_3mo"], "o", label="Three-month trials")
ax.plot(x, ordered["approx_local_sigma_2yr"], "s", label="Two-year approximation")
ax.axhline(3, color="black", linestyle="--", label="Approximate local 3 sigma")
labels = [name + (" *" if failed else "")
          for name, failed in zip(ordered["source"], ordered["n_failed"])]
ax.set_xticks(x, labels, rotation=45, ha="right")
ax.set(ylabel="Approximate local significance (sigma)",
       title="0.1–10 MeV; * marks sources with failed fits")
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "agn_significance_0p1_10_MeV.png", dpi=160)
plt.show()
